# 📦 Chapter 5: Package Management

📖 Book: Chapter 5, pp. 189–235 · ⏱️ 2–3 hours · 🧱 Needs: Ch 3–4 · Level: Intermediate

---

> **🎯 Key idea:** Declarative `pyproject.toml` packaging and `systemd` service units turn ad-hoc scripts into reproducible, self-healing production tools.

> **💡 Tip:** Run cells sequentially top-to-bottom (`Shift + Enter`).

> **⚠️ Warning:** Cells tagged `external` may require extra packages; cloud stacks incur costs if deployed.

## 🔄 Then vs. Now

> **📜 Then (book, ~2020):** `setup.py sdist`, `setup.py bdist_rpm`  
> **🆕 Now (2026):** `pyproject.toml` (PEP 621) + `python -m build`  
> **Why:** `setup.py` CLI commands are deprecated

> **📜 Then (book, ~2020):** `virtualenv` + global `pip`  
> **🆕 Now (2026):** `uv venv` / `python -m venv`, `pipx` / `uv tool`  
> **Why:** Faster, isolated CLI environments

---

## 4. 💡 Conceptual Cards

### 🚨 Real-World Problem
Your helper script started as one file, but now three teammates copy-paste it and hit missing-dependency errors. Worse, someone started the API with `nohup python app.py &`, and it stayed down after a server reboot. Packaging with `pyproject.toml` and supervising with `systemd` solves both problems.

### Concept: PEP 621 `pyproject.toml`
**What:** Standard Python package metadata  
**Why:** Reproducible builds & dependency pinning  
**How:** Parsed by stdlib `tomllib` (3.11+); built by `build`  
**When:** Every shared library or CLI tool

📖 *Book listing:* `hello-world` Package Metadata (pp. 200–206)

**Plain-language analogy:** `pyproject.toml` is a recipe card listing exact ingredient versions so any kitchen (`pip`/`uv`) bakes the identical cake.

**Walkthrough:** The book defines package name, SemVer version, and dependencies for distribution. Modern Python replaces `setup.py` CLI invocations with PEP 621 `pyproject.toml` and `python -m build`.

### Concept: `systemd` Supervision
**What:** Linux init daemon for services  
**Why:** Auto-restarts crashes & streams `journalctl` logs  
**How:** INI unit in `/etc/systemd/system/*.service`  
**When:** Long-running Python APIs & workers

📖 *Book listing:* Pecan HTTP `systemd` Service Unit (pp. 220–232)

**Plain-language analogy:** `systemd` is like an automated building manager: if your Python app trips and falls at 3 AM, `systemd` picks it right back up and logs what happened in `journalctl`.

**Walkthrough:** The book writes an INI unit in `/etc/systemd/system/` with `After=network.target`, absolute `ExecStart`, `WorkingDirectory`, and `StandardOutput=journal` so `journalctl` captures all logs.

## 6. 🟢 Beginner Examples

#### Example 1: Beginner Example 1: Query installed distributions with stdlib importlib.metadata
*Expected output:* Printed labelled output below.

In [1]:
# Beginner Example 1: Query installed distributions with stdlib importlib.metadata
import importlib.metadata as meta

dists = sorted({d.metadata['Name']: d.version for d in meta.distributions()}.items())
print(f'Total installed distributions: {len(dists)}')
for name, ver in dists[:5]:
    print(f'  {name}=={ver}')

Total installed distributions: 100
  Jinja2==3.1.6
  MarkupSafe==3.0.4
  PyYAML==6.0.4
  Pygments==2.21.0
  Send2Trash==2.2.0


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 2: Beginner Example 2: Parse & validate SemVer strings
*Expected output:* Printed labelled output below.

In [2]:
# Beginner Example 2: Parse & validate SemVer strings
import re

SEMVER_RE = re.compile(r'^(?P<major>0|[1-9]\d*)\.(?P<minor>0|[1-9]\d*)\.(?P<patch>0|[1-9]\d*)$')

def parse_semver(ver: str) -> tuple[int, int, int]:
    m = SEMVER_RE.match(ver.strip())
    if not m:
        raise ValueError(f'Invalid SemVer: {ver!r}')
    return int(m.group('major')), int(m.group('minor')), int(m.group('patch'))

print('Parsed 2.4.1:', parse_semver('2.4.1'))

Parsed 2.4.1: (2, 4, 1)


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 3: Beginner Example 3: Parse pyproject.toml using Python 3.11+ stdlib tomllib
*Expected output:* Printed labelled output below.

In [3]:
# Beginner Example 3: Parse pyproject.toml using Python 3.11+ stdlib tomllib
import tomllib

PYPROJECT = '''
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "devops-health"
version = "0.2.0"
requires-python = ">=3.11"
dependencies = ["click>=8.1", "pyyaml>=6.0"]

[project.scripts]
devops-health = "devops_health.cli:main"
'''
parsed = tomllib.loads(PYPROJECT)
print('Project:', parsed['project']['name'], parsed['project']['version'])
print('CLI entry point:', parsed['project']['scripts'])

Project: devops-health 0.2.0
CLI entry point: {'devops-health': 'devops_health.cli:main'}


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

## 7. 🟡 Intermediate Examples: `systemd` Unit Validation & Legacy `setup.py` Comparison

#### Example 4: Intermediate Example 1: Generate & validate an INI systemd service unit
*Expected output:* Printed labelled output below.

In [4]:
# Intermediate Example 1: Generate & validate an INI systemd service unit
import configparser

def render_systemd_unit(name: str, exec_start: str, workdir: str, user: str = 'appuser') -> str:
    return (
        f'[Unit]\nDescription={name}\nAfter=network.target\n\n'
        f'[Service]\nType=simple\nUser={user}\nExecStart={exec_start}\n'
        f'WorkingDirectory={workdir}\nRestart=on-failure\nStandardOutput=journal\n\n'
        f'[Install]\nWantedBy=multi-user.target\n'
    )

unit_txt = render_systemd_unit('devops-api', '/opt/app/.venv/bin/python -m app', '/opt/app')
cp = configparser.ConfigParser()
cp.read_string(unit_txt)
assert cp['Service']['ExecStart'].startswith('/'), 'ExecStart must be absolute!'
print('Validated systemd sections:', cp.sections())

Validated systemd sections: ['Unit', 'Service', 'Install']


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 5: Intermediate Example 2 (Legacy Book Comparison + Fallback): setup.py
*Expected output:* Simulated fallback if optional package is missing.

In [5]:
# Intermediate Example 2 (Legacy Book Comparison + Fallback): setup.py
# Legacy (book, ~2020): setup.py (prefer pyproject.toml above for new projects)
try:
    from setuptools import setup, find_packages
    print('setuptools available for legacy setup.py builds')
except ImportError:
    print('setuptools not installed in kernel; use `uv run --with build python -m build` with pyproject.toml')

setuptools not installed in kernel; use `uv run --with build python -m build` with pyproject.toml


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

## 8. 🚀 Real-World DevOps Applications (Personal DevOps Learning Track)
*Supplementary extension:* **Dependency Pinning Auditor**—scans `requirements.txt` lines to flag unpinned packages (`requests` instead of `requests==2.32.3`) before a production release.

#### Example 6: def find_unpinned(reqs: list[str]) -> list[str]:
*Expected output:* Printed labelled output below.

In [6]:
def find_unpinned(reqs: list[str]) -> list[str]:
    ops = ('==', '>=', '<=', '~=')
    return [
        r.strip() for r in reqs
        if r.strip() and not r.strip().startswith('#') and not any(op in r for op in ops)
    ]

print('Unpinned deps:', find_unpinned(['flask==3.0.3', '# comment', 'requests', 'pyyaml>=6.0']))

Unpinned deps: ['requests']


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 7: Real Tool Demo: systemctl version check (guarded with shutil.which)
*Expected output:* Printed labelled output below.

In [7]:
# Real Tool Demo: systemctl version check (guarded with shutil.which)
import shutil, subprocess
if shutil.which('systemctl'):
    print(subprocess.run(['systemctl', '--version'], capture_output=True, text=True, check=True).stdout.splitlines()[0])
else:
    print('systemctl not installed; skipped')

systemd 259 (259.9-0ubuntu0.1)


**What the output means:** Checks whether the real CLI tool exists on `PATH`; if found, runs a safe read-only/dry-run check, otherwise skips cleanly.

> **🧪 Try it:** Try adding `--help` or inspecting `cp.returncode` to see how exit codes behave.

---

## 9. ⚠️ Common Mistakes & Troubleshooting

- ❌ **Relative `ExecStart` in `systemd`** -> `Executable path is not absolute`. ✅ Use full `.venv/bin/python` path.
- ❌ **Forgetting `systemctl daemon-reload`** -> Unit edits ignored. ✅ Run `sudo systemctl daemon-reload`.
- ❌ **Running `setup.py install` directly** -> Bypasses `pip` resolver. ✅ Use `pip install .` or `python -m build`.

#### Example 8: import importlib.metadata as meta
*Expected output:* Printed labelled output below.

In [8]:
import importlib.metadata as meta
try:
    meta.version('nonexistent-devops-pkg')
except meta.PackageNotFoundError as err:
    print('Caught PackageNotFoundError:', err)

Caught PackageNotFoundError: No package metadata was found for nonexistent-devops-pkg


**What the output means:** Shows the exact exception caught cleanly instead of crashing the whole script.

> **🧪 Try it:** Change the invalid input to a valid one and observe the `except` branch no longer triggers.

## 10. 🛠️ Hands-On Exercises
1. **(Beginner)** `is_pkg_installed(name: str) -> bool`: Return `True` if installed via `importlib.metadata`.
2. **(Beginner)** `bump_patch(ver: str) -> str`: Increment patch version (`'1.2.3'` -> `'1.2.4'`).
3. **(Beginner)** `format_changelog(version: str, date: str, items: list[str]) -> str`: Render Markdown release notes.
4. **(Intermediate)** `validate_systemd_unit(ini_text: str) -> list[str]`: Check `[Unit]`, `[Service]`, `[Install]` and absolute `ExecStart`.
5. **(Intermediate)** `parse_pyproject_deps(toml_text: str) -> list[str]`: Extract `[project].dependencies` using `tomllib`.

## 11. 🏆 Practical DevOps Challenge
`preflight_release(version: str, unit_ini: str, reqs: list[str]) -> dict`: Validate SemVer, `systemd` unit, and pinned dependencies, returning `{'ready': bool, 'errors': list[str]}`.

> **🧪 Try it:** The starter cell below reports `TODO` until you implement the function body.

> **🧪 Try it:** Run the starter cell below—it prints `TODO` until you implement the body.

#### 🧪 Starter Exercise: `Starter Exercise Cell (implement bump_patch and run to verify!)`
*Expected output:* `TODO` message until implemented, then `✅ Starter checks PASSED!`.

In [9]:
# Starter Exercise Cell (implement bump_patch and run to verify!)
def _run_starter_checks():
    assert bump_patch('1.2.3') == '1.2.4'
def bump_patch(ver: str) -> str:
    raise NotImplementedError('Implement bump_patch')

try:
    _run_starter_checks()
    print('✅ Starter checks PASSED!')
except NotImplementedError as e:
    print(f'🧪 TODO: {e}')

🧪 TODO: Implement bump_patch


## 12. 🧠 Knowledge Checks (Inline Q&A for Practice)
**Q1:** What do `MAJOR.MINOR.PATCH` mean in SemVer?

<details>
<summary>✅ Answer 1</summary>

`MAJOR` = breaking changes, `MINOR` = backward-compatible features, `PATCH` = bug fixes.

</details>

**Q2:** Why prefer `pyproject.toml` + `python -m build` over `python setup.py sdist`?

<details>
<summary>✅ Answer 2</summary>

Direct `setup.py` commands are deprecated; `pyproject.toml` (PEP 621) is declarative and tool-agnostic.

</details>

**Q3:** Why must `ExecStart` in `systemd` be an absolute path?

<details>
<summary>✅ Answer 3</summary>

`systemd` does not search `$PATH` for relative executables.

</details>

**Q4:** What is `pipx` (or `uv tool install`) used for?

<details>
<summary>✅ Answer 4</summary>

Installing Python CLI tools in isolated virtualenvs while exposing their binaries globally.

</details>

**Q5:** Which `systemctl` command is required after editing a `.service` file?

<details>
<summary>✅ Answer 5</summary>

`sudo systemctl daemon-reload`.

</details>

## 13. 🔑 Hints & Complete Solutions

<details>
<summary>💭 Hints (click to expand)</summary>

- **Ex 1**: Catch `meta.PackageNotFoundError`.
- **Ex 5**: Use `tomllib.loads(toml_text).get('project', {}).get('dependencies', [])`.

</details>

> **💡 Tip:** Complete solutions are in the collapsed/marked cell below.

<details>
<summary>✅ Complete Solutions (expand/run cell below to verify)</summary>

Run the verified solutions cell below to check all exercise & challenge assertions.

</details>

In [10]:
# Complete Solutions + Assert Verification
import configparser, importlib.metadata as meta, tomllib

def is_pkg_installed(name: str) -> bool:
    try:
        meta.version(name)
        return True
    except meta.PackageNotFoundError:
        return False

def bump_patch(ver: str) -> str:
    maj, min_, pat = parse_semver(ver)
    return f'{maj}.{min_}.{pat + 1}'

def format_changelog(version: str, date: str, items: list[str]) -> str:
    bullets = '\n'.join(f'- {i}' for i in items)
    return f'## [{version}] - {date}\n{bullets}'

def validate_systemd_unit(ini_text: str) -> list[str]:
    errors: list[str] = []
    cp = configparser.ConfigParser()
    cp.read_string(ini_text)
    for sec in ('Unit', 'Service', 'Install'):
        if not cp.has_section(sec):
            errors.append(f'Missing [{sec}]')
    if not cp.get('Service', 'ExecStart', fallback='').startswith('/'):
        errors.append('ExecStart must be absolute')
    return errors

def parse_pyproject_deps(toml_text: str) -> list[str]:
    return tomllib.loads(toml_text).get('project', {}).get('dependencies', [])

def preflight_release(version: str, unit_ini: str, reqs: list[str]) -> dict:
    errors: list[str] = []
    try:
        parse_semver(version)
    except ValueError as e:
        errors.append(str(e))
    errors.extend(validate_systemd_unit(unit_ini))
    for u in find_unpinned(reqs):
        errors.append(f'Unpinned dependency: {u}')
    return {'ready': len(errors) == 0, 'errors': errors}

assert is_pkg_installed('pip') is True
assert bump_patch('1.2.3') == '1.2.4'
assert '## [1.0.0]' in format_changelog('1.0.0', '2026-11-27', ['Initial'])
assert validate_systemd_unit(unit_txt) == []
assert len(parse_pyproject_deps(PYPROJECT)) == 2
assert preflight_release('1.0.0', unit_txt, ['click==8.1.7'])['ready'] is True
print('All Chapter 5 assertions PASSED!')

All Chapter 5 assertions PASSED!


---

## 14. 📌 Chapter Summary & Quick Reference

- `tomllib.loads(txt)` — Parse `pyproject.toml` (e.g. `tomllib.loads(txt)['project']`)
- `importlib.metadata.version()` — Check installed pkg version (e.g. `meta.version('pip')`)
- `journalctl -u svc -fu` — Tail `systemd` logs (e.g. `journalctl -u devops-api --since '-15min'`)

### ✅ What You Can Now Do
- Author and parse PEP 621 `pyproject.toml` manifests using Python 3.11+ `tomllib`.
- Inspect installed packages with `importlib.metadata` and audit unpinned requirements.
- Generate and validate `systemd` `.service` units for long-running Python APIs.

### ➡️ Next Chapter
Chapter 6 builds on this by automating syntax checks, packaging, and deployments inside CI/CD pipelines.